In [1]:
#Python
from pyrosetta import *
from pyrosetta.rosetta import *
from pyrosetta.teaching import *
from pyrosetta.toolbox import *

#Core Includes
#from rosetta.core.kinematics import MoveMap
from pyrosetta.rosetta.core.kinematics import FoldTree
from rosetta.core.pack.task import TaskFactory
from rosetta.core.pack.task import operation
from rosetta.core.simple_metrics import metrics
from rosetta.core.select import residue_selector as selections
from rosetta.core import select
from rosetta.core.select.movemap import *

#Protocol Includes
from rosetta.protocols import minimization_packing as pack_min
from rosetta.protocols import relax as rel
from rosetta.protocols.antibody.residue_selector import CDRResidueSelector
from rosetta.protocols.antibody import *
from rosetta.protocols.loops import *
from rosetta.protocols.relax import FastRelax

#double check that everything is outputting properly in the notebook
print('Checkpoint')


Checkpoint


/tmp/ipykernel_1105/38406194.py:10: UserWarning: Import of 'rosetta' as a top-level module is deprecated and may be removed in 2018, import via 'pyrosetta.rosetta'.
  from rosetta.core.pack.task import TaskFactory


In [16]:
from pyrosetta.rosetta.core.pose import pdbslice
from pyrosetta.rosetta.utility import vector1_unsigned_long

In [3]:
#initializes pyRosetta with the necessary flags and specifications
init('-use_input_sc -input_ab_scheme AHo_Scheme -ignore_unrecognized_res \
     -ignore_zero_occupancy false -load_PDB_components false -relax:default_repeats 2 -no_fconfig')

┌───────────────────────────────────────────────────────────────────────────────┐
│                                  PyRosetta-4                                  │
│               Created in JHU by Sergey Lyskov and PyRosetta Team              │
│               (C) Copyright Rosetta Commons Member Institutions               │
│                                                                               │
│ NOTE: USE OF PyRosetta FOR COMMERCIAL PURPOSES REQUIRES PURCHASE OF A LICENSE │
│          See LICENSE.PyRosetta.md or email license@uw.edu for details         │
└───────────────────────────────────────────────────────────────────────────────┘
PyRosetta-4 2026 [Rosetta PyRosetta4.conda.ubuntu.cxx11thread.serialization.Ubuntu.python310.Release 2026.06+release.1a56185c2592611dec4c9c75ddc9468cd2227c1f 2026-01-30T13:14:27] retrieved from: http://www.pyrosetta.org
core.init: Rosetta version: PyRosetta4.conda.ubuntu.cxx11thread.serialization.Ubuntu.python310.Release r424 2026.06+release.

In [ ]:
#Rename this path to wherever you have saved your pdb file
pose = pose_from_pdb('7K18.pdb')

core.chemical.GlobalResidueTypeSet: Finished initializing fa_standard residue type set.  Created 985 residue types
core.chemical.GlobalResidueTypeSet: Total time to initialize 0.729382 seconds.
core.import_pose.import_pose: File '7K18.pdb' automatically determined to be of type PDB from contents.
core.io.pose_from_sfr.PoseFromSFRBuilder: [ WARNING ] skipping pdb residue b/c it's missing too many mainchain atoms:    3 D BMA BMAA:CtermProteinFull
core.io.pose_from_sfr.PoseFromSFRBuilder: missing:  N
core.io.pose_from_sfr.PoseFromSFRBuilder: missing:  CA
core.io.pose_from_sfr.PoseFromSFRBuilder: missing:  C
core.conformation.Conformation: [ WARNING ] missing heavyatom:  CG  on residue MET 19
core.conformation.Conformation: [ WARNING ] missing heavyatom:  SD  on residue MET 19
core.conformation.Conformation: [ WARNING ] missing heavyatom:  CE  on residue MET 19
core.conformation.Conformation: [ WARNING ] missing heavyatom:  CG  on residue LYS 56
core.conformation.Conformation: [ WARNING ] 

In [12]:
#the original PDB file will have a bunch of stuff (like extra chains of the protein) so we remove that here

print(os.getcwd())
print(pose.sequence())
print(pose.residue(958).name())
print(len(pose.sequence()))
#we want residues 1-111 in our truncated version, just chain A

/home/jcape/coding/pyRosetta_tutorial
VRRAAVKILVHSLFSMLIMCTILTNCVFMAQHDPPPWTKYVEYTFTAIYTFESLVKILARGFCLHAFTFLRDPWNWLDFSVIVMAYTTEFVDGNVSALRTFRVLRALKTISVISGLKTIVGALIQSVKKLADVMVLTVFCLSVFALIGLQLFMGNLRHKCVRNFTELNGTNGSVEASLDVYLNDPANYLLKNGTTDVLLCGNSSDAGTCPEGYRCLKAGENPDHGYTSFDSFAWAFLALFRLMTQDCWERLYQQTLRSAGKIYMIFFMLVIFLGSFYLVNLILAVVAMAYEEQNQATECCPLWMSIKQKVKFVVMDPFADLTITMCIVLNTLFMALEHYNMTAEFEEMLQVGNLVFTGIFTAEMTFKIIALDPYYYFQQGWNIFDSIIVILSLMELGSVLRSFRLLRVFKLAKSWPTLNTLIKIIGNSVGALGNLTLVLAIIVFIFAVVGMQLFGKNYSELRHRISDSGLLPRWHMMDFFHAFLIIFRILCGEWIETMWDCMEVSGQSLCLLVFLLVMVIGNLVVLNLFLALLLSSFGKVWWRLRKTCYRIVEHSWFETFIIFMILLSSGALAFEDIYLEERKTIKVLLEYADKMFTYVFVLEMLLKWVAYGFKKYFTNAWCWLDFLIVDVSLVSLVANTLGFAEMGPIKSLRTLRALRPLRALSRFEGMRVVVNALVGAIPSIMNVLLVCLIFWLIFSIMGVNLFAGKFGRCINQTEGDLPLNYTIVNNKSECESFNVTGELYWTKVKVNFDNVGAGYLALLQVATFKGWMDIMYAAVDSRGYEEQPQWEDNLYMYIYFVVFIIFGSFFTLNLFIGVIIDNFNQQKKKLGGQDIFMTEEQKKYYNAMKKLGSKKPQKPIPRPLNKYQGFIFDIVTKQAFDVTIMFLICLNMVTMMVETDDQSPEKVNILAKINLLFVAIFTGECIVKMAALRHYYFTNSWNIFDFVVVILSIVGTVLSDII

In [ ]:
# Take a new pose of region of interest
# Put pose in a Rosetta vector first
residues = vector1_unsigned_long()
for i in range(900, 1000):
    residues.append(i)

# Create new pose of residues of interest
new_pose = Pose()
pdbslice(new_pose, pose, residues)

# Ensure slices went well
print(testPose.total_residue())
print(testPose.sequence())

DDQSPEKVNILAKINLLFVAIFTGECIVKMAALRHYYFTNSWNIFDFVVVILSIVGTVLSDIIQKYFFSPTLFRVIRLARIGRILRLIRGAKGIRTLLFA


In [22]:
#relax the structure
testPose = Pose()
testPose.assign(new_pose)
print(testPose)

relax = FastRelax()
scorefxn = get_fa_scorefxn()
relax.set_scorefxn(scorefxn)
relax = rosetta.protocols.relax.FastRelax()
relax.constrain_relax_to_start_coords(True)
print(relax)
relax.apply(testPose)

#rename to your desired relaxed structure name
testPose.dump_pdb('test.relax.pdb')

PDB file name: 
Total residues: 100
Sequence: DDQSPEKVNILAKINLLFVAIFTGECIVKMAALRHYYFTNSWNIFDFVVVILSIVGTVLSDIIQKYFFSPTLFRVIRLARIGRILRLIRGAKGIRTLLFA
Fold tree:
FOLD_TREE  EDGE 1 100 -1 
protocols.relax.RelaxScriptManager: Reading relax scripts list from database.
core.scoring.ScoreFunctionFactory: SCOREFUNCTION: ref2015
protocols.relax.RelaxScriptManager: Looking for MonomerRelax2019.txt
protocols.relax.RelaxScriptManager: ================== Reading script file: /home/jcape/miniconda3/envs/mol-dyn/lib/python3.10/site-packages/pyrosetta/database/sampling/relax_scripts/MonomerRelax2019.txt ==================
protocols.relax.RelaxScriptManager: repeat %%nrepeats%%
protocols.relax.RelaxScriptManager: coord_cst_weight 1.0
protocols.relax.RelaxScriptManager: scale:fa_rep 0.040
protocols.relax.RelaxScriptManager: repack
protocols.relax.RelaxScriptManager: scale:fa_rep 0.051
protocols.relax.RelaxScriptManager: min 0.01
protocols.relax.RelaxScriptManager: coord_cst_weight 0.5
protocols.relax.Rela

: 

In [7]:
#USE THIS STEP ONLY IF YOU NEED TO TRIM THE STRUCTURE SOMEWAY SOMEHOW

#FUNCTION FOR TRUNCATING SYSTEM TO ONLY CHAIN A (residues 1-111 here)

#we know that one chain is residues 1-111 because of the print(len(pose.sequence())) above

pyrosetta.rosetta.protocols.grafting.delete_region(pose,112,333)
print(trunc_pose.sequence())
print(pose.annotated_sequence())

#WRITE THE TRUNCATED FILE TO A NEW FILE AND USE THAT GOING FORWARD
pose.dump_pdb('1rwy_trunc.pdb')

protocols.grafting.util: Deleting 222 residues from 112 to 333



ERROR: Pose::delete_residue_range_slow( Size const start, Size const end ): range extends past end(s) of pose!
ERROR:: Exit from: /Volumes/scratch/b3.r/rosetta/commits/rosetta/source/src/core/pose/Pose.cc line: 723


RuntimeError: 

File: /Volumes/scratch/b3.r/rosetta/commits/rosetta/source/src/core/pose/Pose.cc:723
[ ERROR ] UtilityExitException
ERROR: Pose::delete_residue_range_slow( Size const start, Size const end ): range extends past end(s) of pose!



In [ ]:
#WRITE FUNCTION FOR MUTATION
import pyrosetta
from pyrosetta import pose_from_pdb, get_score_function
import os
import random

def pack(pose, posi, amino, scorefxn):

    #Set Reference Pose
    RMSD_calc = pyrosetta.rosetta.core.simple_metrics.metrics.RMSDMetric(relaxPose)
    
    
    # Select Mutate Position
    mut_posi = pyrosetta.rosetta.core.select.residue_selector.ResidueIndexSelector()
    mut_posi.set_index(posi)
    #print(pyrosetta.rosetta.core.select.get_residues_from_subset(mut_posi.apply(pose)))

    # Select Neighbor Position
    nbr_selector = pyrosetta.rosetta.core.select.residue_selector.NeighborhoodResidueSelector()
    nbr_selector.set_focus_selector(mut_posi)
    nbr_selector.set_include_focus_in_subset(True)
    #print(pyrosetta.rosetta.core.select.get_residues_from_subset(nbr_selector.apply(pose)))
    
    # Select No Design Area
    not_design = pyrosetta.rosetta.core.select.residue_selector.NotResidueSelector(mut_posi)
    #print(pyrosetta.rosetta.core.select.get_residues_from_subset(not_design.apply(pose)))

    # The task factory accepts all the task operations
    tf = pyrosetta.rosetta.core.pack.task.TaskFactory()

    # These are pretty standard
    tf.push_back(pyrosetta.rosetta.core.pack.task.operation.InitializeFromCommandline())
    tf.push_back(pyrosetta.rosetta.core.pack.task.operation.IncludeCurrent())
    tf.push_back(pyrosetta.rosetta.core.pack.task.operation.NoRepackDisulfides())

    # Disable Packing
    prevent_repacking_rlt = pyrosetta.rosetta.core.pack.task.operation.PreventRepackingRLT()
    #True indicates here that we are flipping the selection.  So that we are turning off everything but the CDR and its neighbors.
    prevent_subset_repacking = pyrosetta.rosetta.core.pack.task.operation.OperateOnResidueSubset(prevent_repacking_rlt, nbr_selector, True )
    tf.push_back(prevent_subset_repacking)

    # Disable design
    tf.push_back(pyrosetta.rosetta.core.pack.task.operation.OperateOnResidueSubset(
        pyrosetta.rosetta.core.pack.task.operation.RestrictToRepackingRLT(),not_design))

    # Enable design
    aa_to_design = pyrosetta.rosetta.core.pack.task.operation.RestrictAbsentCanonicalAASRLT()
    aa_to_design.aas_to_keep(amino)
    tf.push_back(pyrosetta.rosetta.core.pack.task.operation.OperateOnResidueSubset(aa_to_design, mut_posi))
    
    # Create Packer
    packer = pyrosetta.rosetta.protocols.minimization_packing.PackRotamersMover()
    packer.task_factory(tf)
    print(tf.create_task_and_apply_taskoperations(pose))


    
    #Perform The Move
    if not os.getenv("DEBUG"):
      packer.apply(pose)


#Load the previously-relaxed pose.
#grab your initial structure again
relaxPose = pose_from_pdb('test.relax.pdb')
print(relaxPose.sequence())
#Clone it
original = relaxPose.clone()
scorefxn = get_score_function()
print("\nOld Energy:", scorefxn(original),"\n")

#Input the residue number that you wish to mutate and the 1-letter code 
#If you are substituting multiple, you can just have them listed separately as exampled below
pack(relaxPose, 45, 'A', scorefxn)
#pack(relaxPose, 81, 'D', scorefxn)
print("\nNew Energy:", scorefxn(relaxPose),"\n")

#SAVE THE NEW PDB FILE HERE:
relaxPose.dump_pdb('7K18_e81a.pdb')

#double check that the residue was replaces in the sequence

print(relaxPose.sequence())
print(pose.residue(45).name())

#Set relaxPose back to original 
relaxPose = original.clone()

#Figure out how to minimize the structure or select the rotamers of residues in the region that produces the smallest energy/strain

core.import_pose.import_pose: File '/Users/alecloftus/Desktop/PKHLAB/PV_Reengagement/1rwy_WT/1rwy_wt.pdb' automatically determined to be of type PDB
SMTDLLSAEDIKKAIGAFTAADSFDHKKFFQMVGLKKKSADDVKKVFHILDKDKSGFIEEDELGSILKGFSSDARDLSAKETKTLMAAGDKDGDGKIGVEEFSTLVAESZZ
core.scoring.ScoreFunctionFactory: SCOREFUNCTION: ref2015
basic.io.database: Database file opened: scoring/score_functions/elec_cp_reps.dat
core.scoring.elec.util: Read 40 countpair representative atoms
core.pack.dunbrack.RotamerLibrary: shapovalov_lib_fixes_enable option is true.
core.pack.dunbrack.RotamerLibrary: shapovalov_lib::shap_dun10_smooth_level of 1( aka lowest_smooth ) got activated.
core.pack.dunbrack.RotamerLibrary: Binary rotamer library selected: /Users/alecloftus/miniconda3/envs/AmberTools23/lib/python3.10/site-packages/pyrosetta/database/rotamer/shapovalov/StpDwn_0-0-0/Dunbrack10.lib.bin
core.pack.dunbrack.RotamerLibrary: Using Dunbrack library binary file '/Users/alecloftus/miniconda3/envs/AmberTools23/lib/pytho